# 03 — Model Training
Train all four classifiers with 5-fold CV on the training split and save to disk.

In [ ]:
import pickle, numpy as np, pandas as pd
from signalrag.config import TRAIN_CSV, FEATURE_COLS, LABEL_COL
from signalrag.classifier import (
    build_logistic_regression, build_mlp,
    build_random_forest, build_xgboost
)


In [ ]:
train_df  = pd.read_csv(TRAIN_CSV)
feat_cols = [c for c in FEATURE_COLS if c in train_df.columns]

# Drop rows with any missing signal values (guards against incomplete extraction runs)
n_before  = len(train_df)
train_df  = train_df[feat_cols + [LABEL_COL]].dropna().reset_index(drop=True)
if len(train_df) < n_before:
    print(f"  Dropped {n_before - len(train_df)} rows with missing values")

X_train = train_df[feat_cols].values
y_train = train_df[LABEL_COL].values
pos = int(y_train.sum())
neg = len(y_train) - pos
print(f"Train: {len(y_train)} | {pos} faithful ({y_train.mean()*100:.1f}%)")


In [ ]:
clfs = {
    "lr":  build_logistic_regression(X_train, y_train, feat_cols),
    "mlp": build_mlp(X_train, y_train, feat_cols),
    "rf":  build_random_forest(X_train, y_train, feat_cols),
    "xgb": build_xgboost(X_train, y_train, feat_cols, pos, neg),
}


In [ ]:
import os
os.makedirs("models", exist_ok=True)
for name, clf in clfs.items():
    with open(f"models/{name}.pkl", "wb") as f:
        pickle.dump(clf, f)
    print(f"  Saved models/{name}.pkl")
